In [ ]:
# Notebook 04 - Recommendation Model (step-by-step)
#
# Input: artifacts/features/ from Notebook 03
# Output: artifacts/models/ (SVD model, hybrid config, evaluation results)
#
# Cell 1 - Setup & Load Artifacts
#   Load all feature artifacts, build lookup dicts, verify shapes
#
# Cell 2 - Popularity-Based Recommender
#   Rank by Bayesian popularity_score, fallback for cold-start users
#
# Cell 3 - Content-Based Recommender
#   Similarity lookup from pre-computed top-k neighbors
#   User personalization via high-rated recipe neighbors
#
# Cell 4 - Collaborative Filtering (SVD)
#   User-item sparse matrix -> TruncatedSVD -> predicted ratings
#   Only works for users seen in train
#
# Cell 5 - Constraint Filter
#   Rule-based filter: max_calories, max_minutes, tags_include
#   Wraps around any recommender output
#
# Cell 6 - Hybrid Combiner
#   Weighted: alpha * content_score + (1-alpha) * cf_score
#   Fallback tiers: cold -> popularity, warm -> content, active -> hybrid
#
# Cell 7 - Qualitative Check
#   Spot-check 3-5 sample users (active / warm / cold-start)
#
# Cell 8 - Evaluation on Validation Set
#   Precision@K, Recall@K, NDCG@K, Coverage
#   Cold-start user report
#
# Cell 9 - Save Model Artifacts
#   svd_model.joblib, hybrid_config.json, evaluation_results.json

In [1]:
# === Cell 1 - Setup & Load Artifacts ===

import pandas as pd
import numpy as np
import ast
import json
import joblib
from pathlib import Path
from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

FEATURES_DIR = Path("../artifacts/features")

# ---------- Load artifacts ----------

with open(FEATURES_DIR / "feature_manifest.json") as f:
    manifest = json.load(f)

tfidf_vectorizer = joblib.load(FEATURES_DIR / "tfidf_vectorizer.joblib")
recipe_tfidf_matrix = sparse.load_npz(FEATURES_DIR / "recipe_tfidf_matrix.npz")

recipe_index_map = pd.read_csv(FEATURES_DIR / "recipe_index_map.csv")
recipe_similarity_topk = pd.read_csv(FEATURES_DIR / "recipe_similarity_topk.csv")
recipe_features = pd.read_csv(FEATURES_DIR / "recipe_features.csv")
popularity_features = pd.read_csv(FEATURES_DIR / "popularity_features.csv")

user_features = pd.read_csv(FEATURES_DIR / "user_features.csv")
user_features["favorite_tags"] = user_features["favorite_tags"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else []
)

train_interactions = pd.read_csv(FEATURES_DIR / "train_interactions.csv")
val_interactions = pd.read_csv(FEATURES_DIR / "val_interactions.csv")
train_interactions["date"] = pd.to_datetime(train_interactions["date"], errors="coerce")
val_interactions["date"] = pd.to_datetime(val_interactions["date"], errors="coerce")

# ---------- Load recipe names from processed data ----------

df_recipes = pd.read_csv("../data/processed/recipes_clean.csv", usecols=["id", "name"])
recipe_name_map = dict(zip(df_recipes["id"], df_recipes["name"]))

# ---------- Build lookup dicts ----------

id2idx = dict(zip(recipe_index_map["recipe_id"], recipe_index_map["matrix_index"]))
idx2id = dict(zip(recipe_index_map["matrix_index"], recipe_index_map["recipe_id"]))

train_user_ids = set(train_interactions["user_id"].unique())
train_recipe_ids = set(train_interactions["recipe_id"].unique())

# ---------- Verify shapes ----------

print(f"Manifest version:        {manifest['version']}")
print(f"Build timestamp:         {manifest['build_timestamp']}")
print()
print(f"recipe_tfidf_matrix:     {recipe_tfidf_matrix.shape}")
print(f"recipe_index_map:        {len(recipe_index_map):,} rows")
print(f"recipe_similarity_topk:  {len(recipe_similarity_topk):,} rows")
print(f"recipe_features:         {len(recipe_features):,} rows")
print(f"popularity_features:     {len(popularity_features):,} rows")
print(f"user_features:           {len(user_features):,} rows")
print(f"train_interactions:      {len(train_interactions):,} rows")
print(f"val_interactions:        {len(val_interactions):,} rows")
print()
print(f"Unique train users:      {len(train_user_ids):,}")
print(f"Unique train recipes:    {len(train_recipe_ids):,}")
print(f"id2idx entries:          {len(id2idx):,}")
print(f"recipe_name_map entries: {len(recipe_name_map):,}")
print()
print(f"Sample user_features favorite_tags: {user_features['favorite_tags'].iloc[0]}")
print("\nAll artifacts loaded successfully.")

Manifest version:        1.0
Build timestamp:         2026-04-17T15:02:37.909049

recipe_tfidf_matrix:     (77300, 20000)
recipe_index_map:        77,300 rows
recipe_similarity_topk:  3,865,000 rows
recipe_features:         77,300 rows
popularity_features:     77,300 rows
user_features:           27,021 rows
train_interactions:      559,500 rows
val_interactions:        139,876 rows

Unique train users:      27,021
Unique train recipes:    72,860
id2idx entries:          77,300
recipe_name_map entries: 77,300

Sample user_features favorite_tags: ['time-to-make', 'preparation', 'course']

All artifacts loaded successfully.


In [2]:
# === Cell 2 - Popularity-Based Recommender ===
#
# Xếp hạng recipe theo popularity_score (Bayesian smoothing từ NB03).
# Đây là recommender đơn giản nhất, dùng làm:
#   - Baseline so sánh với các model phức tạp hơn
#   - Fallback cho cold-start users (không có lịch sử)

# Pre-sort popularity một lần, dùng lại cho mọi lần gọi
popularity_ranked = popularity_features.sort_values("popularity_score", ascending=False).reset_index(drop=True)

def recommend_popular(n=10, exclude_ids=None):
    """Return top-n popular recipes, optionally excluding already-seen IDs."""
    candidates = popularity_ranked
    if exclude_ids:
        candidates = candidates[~candidates["recipe_id"].isin(exclude_ids)]
    top = candidates.head(n).copy()
    top["rank"] = range(1, len(top) + 1)
    top["name"] = top["recipe_id"].map(recipe_name_map)
    return top[["rank", "recipe_id", "name", "popularity_score"]].reset_index(drop=True)

# ---------- Quick test ----------
print("Top-10 popular recipes:")
display(recommend_popular(10))

Top-10 popular recipes:


,rank,recipe_id,name,popularity_score
0,1,77497,2 handed kitchen sink tomato sandwich,4.969171
1,2,34753,basic sourdough bread,4.967149
2,3,154351,kittencal s balsamic vinaigrette,4.965450
3,4,78579,pan release professional pan coating better th...,4.965094
4,5,186029,the best creole cajun seasoning mix,4.964216
5,6,58516,hot pepper jelly,4.961464
6,7,244193,absolutely the best new york cheesecake gluten...,4.959104
7,8,55309,caprese salad tomatoes italian marinated tomatoes,4.958252
8,9,31639,7 layer bean dip,4.957364
9,10,25094,my chicken parmigiana,4.957087


In [3]:
# === Cell 3 - Content-Based Recommender ===
# - recommend_content_similar(recipe_id, n): lookup pre-computed top-k neighbors
def recommend_content_similar(recipe_id, n=10):
    """
    Cho 1 recipe_id → trả n recipes có nội dung giống nhất.
    Tra từ bảng pre-computed similarity (NB03 Cell G).
    """
    # Lấy tất cả neighbors của recipe này (tối đa 50)
    neighbors = recipe_similarity_topk[
        recipe_similarity_topk["recipe_id"] == recipe_id
    ].nlargest(n, "similarity").copy()

    # Nếu recipe không có trong bảng (edge case)
    if neighbors.empty:
        return pd.DataFrame(columns=["rank", "recipe_id", "name", "similarity"])

    neighbors = neighbors.reset_index(drop=True)
    neighbors["rank"] = range(1, len(neighbors) + 1)
    neighbors["name"] = neighbors["neighbor_recipe_id"].map(recipe_name_map)

    return neighbors[["rank", "neighbor_recipe_id", "name", "similarity"]].rename(
        columns={"neighbor_recipe_id": "recipe_id"}
    )
# - recommend_for_user_content(user_id, n):
#     Get recipes user rated >= 4 in train
#     Aggregate neighbors, remove already seen, rank by avg similarity
#     Fallback: use favorite_tags or popular if no history
def recommend_for_user_content(user_id, n=10):
    """
    Gợi ý cho user dựa trên nội dung (content-based).
    
    Flow:
      1. Lấy recipes user đã rated >= 4 trong train  (= "liked recipes")
      2. Với mỗi liked recipe, lấy top-50 neighbors từ bảng similarity
      3. Gộp tất cả neighbors, tính trung bình similarity cho mỗi candidate
      4. Loại recipes user đã xem
      5. Trả top-N
      6. Fallback nếu user không có lịch sử
    """

    # === BƯỚC 1: Lấy lịch sử user ===
    # Chỉ lấy từ train_interactions (không dùng val → anti-leakage)
    user_history = train_interactions[train_interactions["user_id"] == user_id]
    
    # Tất cả recipes user đã tương tác (để loại khỏi kết quả)
    seen_ids = set(user_history["recipe_id"].tolist())
    
    # Chỉ giữ recipes user THÍCH (rating >= 4) làm seed
    liked = user_history[user_history["rating"] >= 4]["recipe_id"].tolist()

    # === BƯỚC 2: Fallback nếu không có lịch sử ===
    if len(liked) == 0:
        # User không thích món nào (hoặc cold-start) → trả về popular
        return recommend_popular(n, exclude_ids=seen_ids)

    # === BƯỚC 3: Lấy neighbors của tất cả liked recipes ===
    # Filter bảng similarity: chỉ giữ dòng có recipe_id nằm trong liked
    neighbors = recipe_similarity_topk[
        recipe_similarity_topk["recipe_id"].isin(liked)
    ].copy()

    # === BƯỚC 4: Loại recipes đã xem ===
    neighbors = neighbors[~neighbors["neighbor_recipe_id"].isin(seen_ids)]

    # === BƯỚC 5: Aggregate — Tính điểm cho mỗi candidate ===
    # Một candidate có thể là neighbor của nhiều liked recipes
    # → Tính TRUNG BÌNH similarity (càng giống nhiều món user thích → càng tốt)
    candidate_scores = (
        neighbors
        .groupby("neighbor_recipe_id")["similarity"]
        .agg(["mean", "count"])
        .reset_index()
    )
    candidate_scores.columns = ["recipe_id", "avg_similarity", "matched_count"]

    # Sort: ưu tiên avg_similarity cao, nếu bằng thì matched_count cao hơn
    candidate_scores = candidate_scores.sort_values(
        ["avg_similarity", "matched_count"],
        ascending=[False, False]
    ).head(n).reset_index(drop=True)

    # === BƯỚC 6: Format kết quả ===
    candidate_scores["rank"] = range(1, len(candidate_scores) + 1)
    candidate_scores["name"] = candidate_scores["recipe_id"].map(recipe_name_map)
    candidate_scores["source"] = "content"

    return candidate_scores[["rank", "recipe_id", "name", "avg_similarity", "matched_count", "source"]]

# ---------- Quick test ----------

# Test hàm 1: Tìm 5 món giống "arriba baked winter squash" (id=137739)
print("=== Similar to Winter Squash ===")
display(recommend_content_similar(137739, n=5))

# Test hàm 2: Gợi ý cho user active
sample_active = user_features[user_features["activity_level"] == "active"]["user_id"].iloc[0]
print(f"\n=== Content recs for active user {sample_active} ===")
display(recommend_for_user_content(sample_active, n=10))

# Test hàm 2: Gợi ý cho user cold-start (không có trong train)
cold_users = set(val_interactions["user_id"]) - train_user_ids
sample_cold = list(cold_users)[0]
print(f"\n=== Content recs for cold-start user {sample_cold} ===")
display(recommend_for_user_content(sample_cold, n=5))

=== Similar to Winter Squash ===


,rank,recipe_id,name,similarity
0,1,235285,mexican squash,0.289120
1,2,216993,sweet and spicy yams,0.286754
2,3,189352,maple sweet dumpling squash,0.282295
3,4,267785,parmesan butternut squash gratin,0.277842
4,5,30377,spiced baked acorn squash,0.262536



=== Content recs for active user 1533 ===


,rank,recipe_id,name,avg_similarity,matched_count,source
0,1,46277,no salt seasoning,0.702102,1,content
1,2,68017,italian seasoning,0.597112,1,content
2,3,4029,herb blend for boursin cream cheese,0.583898,1,content
3,4,48555,memphis rub,0.569698,1,content
4,5,83902,montreal steak seasoning,0.551078,1,content
5,6,14701,stove top scalloped potatoes,0.538989,1,content
6,7,83903,beef steak seasoning mix,0.538385,1,content
7,8,52711,kansas city dry rub,0.528090,1,content
8,9,8286,never fail meringue,0.520701,1,content
9,10,118677,whipping cream,0.513861,1,content



=== Content recs for cold-start user 1925124 ===


,rank,recipe_id,name,popularity_score
0,1,77497,2 handed kitchen sink tomato sandwich,4.969171
1,2,34753,basic sourdough bread,4.967149
2,3,154351,kittencal s balsamic vinaigrette,4.965450
3,4,78579,pan release professional pan coating better th...,4.965094
4,5,186029,the best creole cajun seasoning mix,4.964216


In [4]:
# === Cell 4 - Collaborative Filtering (SVD) ===
# - Build user-item sparse matrix from train_interactions (centered by user mean)
# ─── A1: Tạo mapping user_id → row index ───
# (recipe đã có id2idx từ Cell 1, chỉ cần tạo cho user)

train_user_list = sorted(train_interactions["user_id"].unique())
uid2row = {uid: i for i, uid in enumerate(train_user_list)}
row2uid = {i: uid for uid, i in uid2row.items()}

n_users = len(train_user_list)       # 27,021
n_items = recipe_tfidf_matrix.shape[0]  # 77,300

print(f"Matrix shape sẽ là: {n_users:,} users × {n_items:,} items")

# ─── A2: Build sparse matrix ───

rows = []    # user indices
cols = []    # item indices
vals = []    # ratings

for _, row in train_interactions.iterrows():
    u = uid2row.get(row["user_id"])
    i = id2idx.get(row["recipe_id"])
    if u is not None and i is not None:
        rows.append(u)
        cols.append(i)
        vals.append(row["rating"])

R = sparse.csr_matrix(
    (vals, (rows, cols)),
    shape=(n_users, n_items),
    dtype=np.float32
)

print(f"R shape: {R.shape}")
print(f"R nnz:   {R.nnz:,}")
print(f"Sparsity: {1 - R.nnz / (R.shape[0] * R.shape[1]):.4%}")
# ─── A3: Trừ đi rating trung bình của mỗi user ───

# Tính mean rating cho mỗi user (chỉ tính trên ô khác 0)
user_ratings_sum = np.array(R.sum(axis=1)).flatten()     # tổng rating mỗi user
user_ratings_count = np.array(R.getnnz(axis=1)).flatten() # số ô khác 0 mỗi user
user_means = np.divide(
    user_ratings_sum,
    user_ratings_count,
    out=np.zeros_like(user_ratings_sum),
    where=user_ratings_count != 0
)

# Trừ mean khỏi từng ô có giá trị
R_centered = R.copy().astype(np.float32)
for i in range(n_users):
    start = R_centered.indptr[i]
    end = R_centered.indptr[i + 1]
    R_centered.data[start:end] -= user_means[i]

print(f"User means range: [{user_means.min():.2f}, {user_means.max():.2f}]")
print(f"R_centered data range: [{R_centered.data.min():.2f}, {R_centered.data.max():.2f}]")
# - Fit TruncatedSVD(n_components=50)
# ─── B: Fit TruncatedSVD ───

N_COMPONENTS = 50  # số latent factors

svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
U_reduced = svd.fit_transform(R_centered)  # (27021, 50)
VT = svd.components_                        # (50, 77300)

explained = svd.explained_variance_ratio_.sum()
print(f"U_reduced shape: {U_reduced.shape}")
print(f"VT shape:        {VT.shape}")
print(f"Explained variance: {explained:.2%}")
# - recommend_cf(user_id, n): reconstruct predicted ratings, exclude seen, return top-N
# ─── C: Recommend function ───

# Pre-build: set recipes mỗi user đã xem (để loại khỏi kết quả)
# - Only works for users present in train set
user_seen = {}
for uid, group in train_interactions.groupby("user_id"):
    row_idx = uid2row.get(uid)
    if row_idx is not None:
        item_indices = set()
        for rid in group["recipe_id"]:
            idx = id2idx.get(rid)
            if idx is not None:
                item_indices.add(idx)
        user_seen[row_idx] = item_indices


def recommend_cf(user_id, n=10):
    """
    Collaborative Filtering recommendation cho 1 user.
    
    Flow:
      1. Lấy user vector từ U_reduced
      2. Nhân với VT → predicted ratings cho tất cả recipes
      3. Cộng lại user_mean (vì đã center trước đó)
      4. Loại recipes đã xem
      5. Trả top-N
    """

    # Kiểm tra user có trong train không
    row_idx = uid2row.get(user_id)
    if row_idx is None:
        return pd.DataFrame(columns=["rank", "recipe_id", "name", "predicted_rating", "source"])

    # === BƯỚC 1+2: Dự đoán rating cho TẤT CẢ recipes ===
    # U_reduced[row_idx] có shape (50,)
    # VT có shape (50, 77300)
    # Dot product → (77300,) = predicted rating (centered) cho mỗi recipe
    predicted_centered = U_reduced[row_idx] @ VT  # shape: (77300,)

    # === BƯỚC 3: Cộng lại user mean ===
    predicted = predicted_centered + user_means[row_idx]

    # === BƯỚC 4: Loại recipes đã xem ===
    seen = user_seen.get(row_idx, set())
    for idx in seen:
        predicted[idx] = -np.inf  # đặt = -inf để không bao giờ được chọn

    # === BƯỚC 5: Lấy top-N ===
    # argpartition nhanh O(n), không cần sort toàn bộ 77K
    top_indices = np.argpartition(predicted, -n)[-n:]
    top_indices = top_indices[np.argsort(predicted[top_indices])[::-1]]

    results = []
    for rank, idx in enumerate(top_indices, 1):
        rid = idx2id.get(idx)
        results.append({
            "rank": rank,
            "recipe_id": rid,
            "name": recipe_name_map.get(rid, "?"),
            "predicted_rating": round(float(predicted[idx]), 3),
            "source": "cf"
        })

    return pd.DataFrame(results)

# ---------- Quick test ----------

# Sanity check: SVD đã train
print(f"SVD components: {svd.n_components}")
print(f"Explained variance: {svd.explained_variance_ratio_.sum():.2%}")
print(f"U_reduced shape: {U_reduced.shape}")
print(f"VT shape: {VT.shape}")
print()

# Test active user
sample_active = user_features[user_features["activity_level"] == "active"]["user_id"].iloc[0]
print(f"=== CF recs for active user {sample_active} ===")
display(recommend_cf(sample_active, n=10))

# Test cold-start (phải trả empty)
cold_users = set(val_interactions["user_id"]) - train_user_ids
sample_cold = list(cold_users)[0]
print(f"\n=== CF recs for cold-start user {sample_cold} ===")
result = recommend_cf(sample_cold, n=5)
print(f"Empty (expected): {result.empty}")

Matrix shape sẽ là: 27,021 users × 77,300 items
R shape: (27021, 77300)
R nnz:   559,500
Sparsity: 99.9732%
User means range: [1.00, 5.00]
R_centered data range: [-3.97, 2.83]
U_reduced shape: (27021, 50)
VT shape:        (50, 77300)
Explained variance: 9.86%
SVD components: 50
Explained variance: 9.86%
U_reduced shape: (27021, 50)
VT shape: (50, 77300)

=== CF recs for active user 1533 ===


,rank,recipe_id,name,predicted_rating,source
0,1,54257,yes virginia there is a great meatloaf,4.870,cf
1,2,81853,easy crock pot macaroni and cheese,4.864,cf
2,3,12522,steven s world famous to die for sour cream ch...,4.860,cf
3,4,62236,chocolate cottage cheese,4.859,cf
4,5,37336,melt in your mouth chicken breasts,4.857,cf
5,6,32204,whatever floats your boat brownies,4.857,cf
6,7,75302,mrs geraldine s ground beef casserole,4.857,cf
7,8,9836,slow cooker cheesy chicken,4.857,cf
8,9,224211,honey mustard crock pot chicken,4.856,cf
9,10,83268,3 packet roast,4.856,cf



=== CF recs for cold-start user 1925124 ===
Empty (expected): True


In [ ]:
# === Cell 5 - Constraint Filter ===
# - apply_constraints(candidate_ids, max_calories, max_minutes, tags_include)
# - Filters candidate recipes by rule-based conditions
# - Wraps around any recommender output before returning to user

In [ ]:
# === Cell 6 - Hybrid Combiner ===
# - hybrid_score = alpha * content_score_norm + (1-alpha) * cf_score_norm
# - Fallback tiers:
#     Cold-start (no train history)  -> Popularity + Content from tags
#     Warm (< 5 interactions)        -> Content-heavy (alpha=0.7)
#     Active (>= 5 interactions)     -> Balanced hybrid (alpha=0.5)
# - recommend_hybrid(user_id, n, alpha, constraints)
# - Returns results with source tag: popularity / content / cf / hybrid

In [ ]:
# === Cell 7 - Qualitative Check ===
# - Pick 3-5 sample users: 1 active, 1 warm, 1 cold-start
# - Print recommendations from each model + hybrid
# - Compare: recipe name, score, source

In [ ]:
# === Cell 8 - Evaluation on Validation Set ===
# - Ground truth: recipes rated >= 4 per user in val_interactions
# - Predicted: top-K from each model
# - Metrics: Precision@K, Recall@K, NDCG@K, Coverage
# - Comparison table: Popularity vs Content vs CF vs Hybrid
# - Separate cold-start user report

In [ ]:
# === Cell 9 - Save Model Artifacts ===
# - Save to artifacts/models/:
#     svd_model.joblib
#     user_index_map.csv
#     hybrid_config.json (alpha, tiers, constraint defaults)
#     evaluation_results.json